# SoilSense: IoT + ML soil fertility project

Run the cells from top to bottom (**Kernel → Restart & Run All**). The whole notebook takes about a minute.

```
SoilSense: IoT-driven ML framework for soil fertility classification
and counterfactual fertilizer recommendation (B.Tech CSE student project).

Pipeline
  1. Simulate an hourly 8-in-1 IoT soil-probe stream for many fields
     (temperature, moisture, EC, salinity, pH, N, P, K) with realistic
     sensor faults: Gaussian noise, slow calibration drift, spikes and gaps.
  2. Clean the stream (range check -> Hampel filter -> interpolation)
     and aggregate it to one feature vector per field per day.
  3. Label each day Low / Medium / High fertility from a weighted
     agronomic index computed on the TRUE (noise-free) soil state.
  4. Compare five classifiers with grouped, stratified 5-fold CV.
  5. Rank features with permutation importance.
  6. Raise rule-based irrigation / nitrogen alerts (corrected version
     of the seven-day rule in Shahab et al., 2025).
  7. Search counterfactual recommendations: the smallest change to
     controllable soil properties that moves a field to 'High'.

Run:  python soilsense.py          (about 30-60 s on a laptop)
Needs: numpy, pandas, scikit-learn, matplotlib
```

## 0. Install libraries (only needed once)

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib

## Imports

In [ ]:
%matplotlib inline
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.inspection import permutation_importance

## Settings
Change these to experiment: dataset size, optimal bands, index weights and sensor noise.

In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
rng = np.random.default_rng(SEED)

N_FIELDS, N_DAYS, HOURS = 40, 120, 24
FEATURES = ["temp", "moisture", "ec", "salinity", "ph", "N", "P", "K"]
CLASSES = ["Low", "Medium", "High"]

# Physical limits of the probe (from the 8-in-1 sensor datasheet, Shahab et al.)
LIMITS = {"temp": (-40, 85), "moisture": (0, 100), "ec": (0, 10000),
          "salinity": (0, 10000), "ph": (0, 14), "N": (0, 200),
          "P": (0, 200), "K": (0, 200)}

# Optimal bands used by the fertility index (illustrative; replace with
# Soil Health Card norms for the crop and region being studied).
OPTIMAL = {"N": (80, 140), "P": (15, 30), "K": (110, 190),
           "ph": (6.0, 7.5), "ec": (200, 1500), "moisture": (60, 85)}
WEIGHTS = {"N": 0.25, "P": 0.20, "K": 0.20, "ph": 0.15, "ec": 0.10, "moisture": 0.10}

# Sensor noise (standard deviation per hourly reading)
NOISE = {"temp": 0.5, "moisture": 2.5, "ec": 60, "salinity": 60,
         "ph": 0.08, "N": 4.0, "P": 1.0, "K": 5.0}

## 1. Simulate the IoT stream

In [ ]:
def simulate_true_state():
    """Daily true soil state for every field (no sensor error)."""
    rows = []
    for f in range(N_FIELDS):
        # Field baselines: wide enough to produce all three classes
        base = {"N": rng.uniform(45, 135), "P": rng.uniform(8, 30),
                "K": rng.uniform(80, 185), "ph": rng.uniform(5.4, 8.6),
                "ec": rng.uniform(400, 2600)}
        moisture, n_level = rng.uniform(55, 85), base["N"]
        for d in range(N_DAYS):
            # Moisture: evaporation, random rain, irrigation when very dry
            rain = rng.random() < 0.12
            moisture += -rng.uniform(0.8, 2.2) + (rng.uniform(8, 20) if rain else 0)
            if moisture < 50 and rng.random() < 0.5:
                moisture += rng.uniform(15, 25)            # farmer irrigates
            moisture = float(np.clip(moisture, 35, 98))
            # Nitrogen: slow crop uptake plus occasional top-dressing
            n_level += -rng.uniform(0.1, 0.6)
            if d in (30, 75) and rng.random() < 0.6:
                n_level += rng.uniform(15, 30)
            n_level = float(np.clip(n_level, 20, 190))
            rows.append({
                "field": f, "day": d,
                "temp": 31 + 2 * np.sin(2 * np.pi * d / 120) + rng.normal(0, 0.6),
                "moisture": moisture,
                "ec": base["ec"] + rng.normal(0, 40),
                "ph": base["ph"] + rng.normal(0, 0.05),
                "N": n_level,
                "P": base["P"] + rng.normal(0, 0.5),
                "K": base["K"] + rng.normal(0, 2.0),
            })
    df = pd.DataFrame(rows)
    df["salinity"] = 0.9 * df["ec"] + rng.normal(0, 30, len(df))
    return df


def simulate_sensor_stream(true_daily):
    """Hourly sensor readings = truth + noise + drift + spikes + gaps."""
    hourly = true_daily.loc[true_daily.index.repeat(HOURS)].reset_index(drop=True)
    hourly["hour"] = np.tile(np.arange(HOURS), len(true_daily))
    t = hourly["day"] * HOURS + hourly["hour"]
    for col in FEATURES:
        # Field-specific calibration drift: about 1.5 noise-SDs by season end
        drift_rate = rng.normal(0, NOISE[col] * 1.5 / (N_DAYS * HOURS), N_FIELDS)
        drift = drift_rate[hourly["field"]] * t
        hourly[col] = hourly[col] + rng.normal(0, NOISE[col], len(hourly)) + drift
        spikes = rng.random(len(hourly)) < 0.01           # 1 % spikes
        hourly.loc[spikes, col] += rng.choice([-1, 1], spikes.sum()) * NOISE[col] * 15
        gaps = rng.random(len(hourly)) < 0.03             # 3 % dropped packets
        hourly.loc[gaps, col] = np.nan
    return hourly

## 2. Cleaning and daily aggregation

In [ ]:
def hampel(series, window=7, n_sigma=3.0):
    """Replace points far from the rolling median with that median."""
    med = series.rolling(window, center=True, min_periods=1).median()
    mad = (series - med).abs().rolling(window, center=True, min_periods=1).median()
    outlier = (series - med).abs() > n_sigma * 1.4826 * mad
    return series.mask(outlier, med)


def clean_and_aggregate(hourly, use_filter=True):
    df = hourly.copy()
    for col in FEATURES:
        lo, hi = LIMITS[col]
        df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan     # range check
        if use_filter:
            df[col] = df.groupby("field")[col].transform(hampel)
            df[col] = df.groupby("field")[col].transform(
                lambda s: s.interpolate(limit_direction="both"))
    daily = df.groupby(["field", "day"])[FEATURES].mean().reset_index()
    return daily.dropna()

## 3. Fertility index and labels

In [ ]:
def band_score(x, lo, hi):
    """1 inside the optimal band, falling linearly to 0 at 50 % outside it."""
    width = (hi - lo) * 0.5 + 1e-9
    below = np.clip(1 - (lo - x) / width, 0, 1)
    above = np.clip(1 - (x - hi) / width, 0, 1)
    return np.where(x < lo, below, np.where(x > hi, above, 1.0))


def fertility_index(df):
    return sum(w * band_score(df[k].values, *OPTIMAL[k]) for k, w in WEIGHTS.items())


def label_fertility(index, label_noise=0.03):
    y = np.digitize(index, [0.70, 0.85])                  # 0 Low, 1 Medium, 2 High
    flip = rng.random(len(y)) < label_noise               # lab / annotation error
    y[flip] = rng.integers(0, 3, flip.sum())
    return y

## 4. Model comparison with grouped cross-validation

In [ ]:
def get_models():
    return {
        "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight="balanced",
                                                random_state=SEED),
        "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
        "SVM (RBF)": make_pipeline(StandardScaler(), SVC(C=10, gamma="scale", class_weight="balanced",
                                                         probability=True, random_state=SEED)),
        "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced",
                                                n_jobs=-1, random_state=SEED),
        "Gradient Boosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                                            random_state=SEED),
    }


def evaluate(X, y, groups, models, n_splits=5):
    """Folds keep every field's days together, so no field leaks into the test fold."""
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    results, folds = [], list(cv.split(X, y, groups))
    for name, model in models.items():
        acc, f1 = [], []
        for tr, te in folds:
            model.fit(X.iloc[tr], y[tr])
            pred = model.predict(X.iloc[te])
            acc.append(accuracy_score(y[te], pred))
            f1.append(f1_score(y[te], pred, average="macro"))
        results.append({"model": name, "acc_mean": np.mean(acc), "acc_sd": np.std(acc),
                        "f1_mean": np.mean(f1), "f1_sd": np.std(f1)})
        print(f"  {name:<18} acc {np.mean(acc):.3f} ± {np.std(acc):.3f}   "
              f"macro-F1 {np.mean(f1):.3f} ± {np.std(f1):.3f}")
    return pd.DataFrame(results), folds

## 6. Rule-based alerts (corrected seven-day rule)

In [ ]:
def rule_alerts(daily, fc_threshold=60, n_floor=80, n_drop=0.15, k_days=3, n_cooldown=14):
    """Irrigate after k consecutive dry days; fertilize when N stays below
    n_floor for k days or falls more than n_drop over a week.
    Fixes two errors in the published pseudocode: the dry-day counter is
    incremented (+1, not /1) and the N trigger is relative, because an
    absolute 100 mg/kg drop can never occur when readings span 71-103.
    A cooldown stops repeat N alerts while applied fertilizer takes effect."""
    alerts = []
    for f, g in daily.sort_values("day").groupby("field"):
        dry = low_n = 0
        last_n = -n_cooldown
        n_week = g["N"].rolling(7, min_periods=7)
        for (_, row), n_max in zip(g.iterrows(), n_week.max()):
            dry = dry + 1 if row["moisture"] < fc_threshold else 0
            low_n = low_n + 1 if row["N"] < n_floor else 0
            if dry >= k_days:
                alerts.append((f, int(row["day"]), "IRRIGATE"))
                dry = 0
            dropped = pd.notna(n_max) and row["N"] < (1 - n_drop) * n_max
            if (low_n >= k_days or dropped) and row["day"] - last_n >= n_cooldown:
                alerts.append((f, int(row["day"]), "APPLY_N"))
                low_n, last_n = 0, row["day"]
    return pd.DataFrame(alerts, columns=["field", "day", "action"])

## 7. Counterfactual recommendation

In [ ]:
ACTIONABLE = {"N": (0, 60), "P": (0, 15), "K": (0, 60),     # only additions
              "ph": (-1.5, 1.0), "moisture": (0, 25)}        # lime/gypsum, irrigation


def counterfactual(model, x, scale, target=2, p_min=0.6, n_samples=4000):
    """Random search for the cheapest change (normalised Manhattan distance)
    to controllable features that the model predicts as the target class."""
    cand = pd.DataFrame(np.repeat(x.values[None, :], n_samples, 0), columns=x.index)
    deltas = {}
    for k, (lo, hi) in ACTIONABLE.items():
        keep = rng.random(n_samples) < 0.6                 # sparse changes
        deltas[k] = np.where(keep, rng.uniform(lo, hi, n_samples), 0.0)
        cand[k] = cand[k] + deltas[k]
    prob = model.predict_proba(cand[x.index])[:, target]
    cost = sum(np.abs(deltas[k]) / scale[k] for k in ACTIONABLE)
    ok = prob >= p_min
    if not ok.any():
        return None
    best = np.argmin(np.where(ok, cost, np.inf))
    return {k: round(float(deltas[k][best]), 2) for k in ACTIONABLE
            if abs(deltas[k][best]) > 1e-6}, float(prob[best]), float(cost[best])


def describe(change):
    """Plain-language advice. 1 mg/kg in a 15 cm layer ~ 1.95 kg/ha (bulk
    density 1.3 g/cm3); treat kg/ha as indicative, not a dose."""
    text = []
    for k, v in change.items():
        if k in ("N", "P", "K"):
            text.append(f"raise available {k} by {v:.0f} mg/kg (~{v*1.95:.0f} kg/ha {k})")
        elif k == "ph":
            text.append(f"{'raise' if v > 0 else 'lower'} pH by {abs(v):.1f} "
                        f"({'lime' if v > 0 else 'gypsum/organic matter'})")
        elif k == "moisture":
            text.append(f"irrigate to raise moisture by {v:.0f} points")
    return "; ".join(text)

## Figures

In [ ]:
INK, MUTED, ACCENT, SOFT = "#1f2933", "#6b7280", "#2563eb", "#dbe7fb"


def figure_architecture(path):
    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.set_xlim(0, 11); ax.set_ylim(0, 3.2); ax.axis("off")
    boxes = [
        ("Sensing layer", "8-in-1 soil probe\nRS485 to ESP32\nhourly readings"),
        ("Edge / cloud", "Range check\nHampel filter\ngap interpolation\ndaily means"),
        ("Learning layer", "5 classifiers\ngrouped 5-fold CV\npermutation\nimportance"),
        ("Decision layer", "Rule alerts\n(irrigate, apply N)\ncounterfactual\nsearch"),
        ("Farmer app", "Fertility class\nplain-language advice\nindicative doses"),
    ]
    w, gap, y0, h = 1.85, 0.32, 0.55, 2.3
    for i, (title, body) in enumerate(boxes):
        x = 0.25 + i * (w + gap)
        ax.add_patch(FancyBboxPatch((x, y0), w, h, boxstyle="round,pad=0.02,rounding_size=0.12",
                                    fc=SOFT if i in (2, 3) else "white", ec=ACCENT, lw=1.4))
        ax.text(x + w / 2, y0 + h - 0.3, title, ha="center", va="center",
                fontsize=10.5, weight="bold", color=INK)
        ax.text(x + w / 2, y0 + h / 2 - 0.2, body, ha="center", va="center",
                fontsize=8.4, color=INK, linespacing=1.5)
        if i < len(boxes) - 1:
            ax.annotate("", xy=(x + w + gap - 0.03, y0 + h / 2), xytext=(x + w + 0.03, y0 + h / 2),
                        arrowprops=dict(arrowstyle="-|>", color=MUTED, lw=1.4))
    ax.text(5.5, 0.18, "Shaded layers are implemented in software in this project; "
            "the sensing layer is simulated.", ha="center", fontsize=8.5, color=MUTED)
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def figure_results(res, imp, path):
    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1, 1]})
    res = res.sort_values("f1_mean")
    a.errorbar(res["f1_mean"], res["model"], xerr=res["f1_sd"], fmt="o", ms=8,
               color=ACCENT, ecolor=MUTED, capsize=4, lw=1.5)
    for v, sd, m in zip(res["f1_mean"], res["f1_sd"], res["model"]):
        a.text(v + sd + 0.006, m, f"{v:.3f}", va="center", fontsize=9, color=INK)
    a.set_xlabel("Macro-F1 (mean ± SD over 5 grouped folds)", color=INK)
    a.set_title("(a) Classifier comparison", loc="left", fontsize=11, color=INK)
    names = {"ph": "pH", "ec": "EC", "temp": "Temperature", "moisture": "Moisture",
             "salinity": "Salinity", "N": "Nitrogen", "P": "Phosphorus", "K": "Potassium"}
    imp = imp.sort_values("mean").assign(feature=lambda d: d["feature"].map(names))
    b.barh(imp["feature"], imp["mean"], xerr=imp["sd"], color=ACCENT,
           ecolor=MUTED, height=0.55, capsize=3)
    b.set_xlabel("Drop in macro-F1 when feature is shuffled", color=INK)
    b.set_title("(b) Permutation importance, best model", loc="left", fontsize=11, color=INK)
    for ax in (a, b):
        ax.grid(axis="x", color="#e5e7eb", lw=0.8); ax.set_axisbelow(True)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
        ax.tick_params(colors=INK)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.close(fig)

## Main pipeline

In [ ]:
def main():
    print("1. Simulating sensor stream ...")
    truth = simulate_true_state()
    hourly = simulate_sensor_stream(truth)
    print(f"   {len(hourly):,} hourly readings, "
          f"{hourly[FEATURES].isna().mean().mean():.1%} missing")

    print("2. Cleaning and aggregating ...")
    daily = clean_and_aggregate(hourly).merge(
        truth[["field", "day"]], on=["field", "day"])
    truth_aligned = truth.set_index(["field", "day"]).loc[
        list(zip(daily["field"], daily["day"]))].reset_index()

    print("3. Labelling fertility ...")
    y = label_fertility(fertility_index(truth_aligned))
    print("   class counts:", dict(zip(CLASSES, np.bincount(y, minlength=3))))
    X, groups = daily[FEATURES], daily["field"].values

    print("4. Comparing classifiers (StratifiedGroupKFold, k=5) ...")
    res, folds = evaluate(X, y, groups, get_models())
    best_name = res.sort_values("f1_mean").iloc[-1]["model"]
    print(f"   best model: {best_name}")

    # Ablation: same best model on unfiltered data
    raw_daily = clean_and_aggregate(hourly, use_filter=False)
    raw = raw_daily.merge(daily[["field", "day"]], on=["field", "day"])
    keep = daily.set_index(["field", "day"]).index.isin(raw.set_index(["field", "day"]).index)
    abl, _ = evaluate(raw[FEATURES], y[keep], raw["field"].values,
                      {best_name + " (no filter)": get_models()[best_name]})

    # Detailed report on the last fold
    best = get_models()[best_name]
    tr, te = folds[-1]
    best.fit(X.iloc[tr], y[tr])
    pred = best.predict(X.iloc[te])
    print(classification_report(y[te], pred, target_names=CLASSES, digits=3))
    print("   confusion matrix:\n", confusion_matrix(y[te], pred))

    print("5. Permutation importance ...")
    pi = permutation_importance(best, X.iloc[te], y[te], scoring="f1_macro",
                                n_repeats=10, random_state=SEED, n_jobs=-1)
    imp = pd.DataFrame({"feature": FEATURES, "mean": pi.importances_mean,
                        "sd": pi.importances_std})
    print(imp.sort_values("mean", ascending=False).to_string(index=False))

    print("6. Rule-based alerts ...")
    alerts = rule_alerts(daily)
    print(alerts["action"].value_counts().to_string())

    print("7. Counterfactual recommendations ...")
    best.fit(X, y)                                       # final model on all data
    scale = X.std()
    low_idx = np.where(best.predict(X) == 0)[0]
    examples = rng.choice(low_idx, size=min(5, len(low_idx)), replace=False)
    found = 0
    for i in examples:
        out = counterfactual(best, X.iloc[i], scale)
        tag = f"field {daily['field'].iat[i]:>2}, day {daily['day'].iat[i]:>3}"
        if out is None:
            print(f"   {tag}: no feasible change within limits")
            continue
        found += 1
        change, p, cost = out
        print(f"   {tag}: P(High)={p:.2f}, cost={cost:.2f} -> {describe(change)}")
    success = np.mean([counterfactual(best, X.iloc[i], scale) is not None
                       for i in rng.choice(low_idx, size=min(100, len(low_idx)), replace=False)])
    print(f"   feasible counterfactuals for 100 Low days: {success:.0%}")

    figure_architecture("fig1_architecture.png")
    figure_results(res, imp, "fig2_results.png")
    res.to_csv("model_results.csv", index=False)
    alerts.to_csv("rule_alerts.csv", index=False)
    print("Saved fig1_architecture.png, fig2_results.png, model_results.csv, rule_alerts.csv")

## Run everything
Prints each stage and saves two figures and two CSV files in the notebook's folder.

In [ ]:
main()

## Figure 1: system architecture

In [ ]:
from IPython.display import Image, display
display(Image("fig1_architecture.png"))

## Figure 2: model comparison and feature importance

In [ ]:
display(Image("fig2_results.png"))

## Results tables

In [ ]:
pd.read_csv("model_results.csv").round(3)

In [ ]:
alerts = pd.read_csv("rule_alerts.csv")
alerts.groupby("action").size()